# Agent 定义：智能客服助手

### 核心定位
基于公司知识库的智能助手，能够理解客户邮件并检索专业知识，生成**高质量回复草稿**供人工审核。

### 关键词
* **Knowledge-grounded**：基于知识库，回答必须有据可查。
* **Assistive**：辅助定位，作为客服插件而非替代人工。
* **Draft-first**：先生成草稿，人工审核/修改后发出。
* **Reliable**：回复内容可控、可解释、符合公司口径。

Where Chunking Fits in the RAG Pipeline
A typical RAG system consists of:
Indexing — Convert documents into vector embeddings and store them in a vector database.
Retrieval — Query the database for the most relevant chunks.
Augmentation — Inject retrieved chunks into the LLM prompt.
Generation — Prompt the LLM to produce a final, context-informed response.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
kb_data = [
    {
        "doc_id": 1,
        "title": "Antibody X Storage",
        "source": "product_doc",
        "text": "Antibody X should be stored at -20°C. Avoid repeated freeze-thaw cycles."
    },
    {
        "doc_id": 2,
        "title": "Shipping Policy",
        "source": "company_doc",
        "text": "Most products are shipped on dry ice. Delivery timing depends on destination and carrier."
    },
    {
        "doc_id": 3,
        "title": "Company Introduction",
        "source": "company_doc",
        "text": "Promab provides antibodies, proteins, and related biotech research products."
    },
    {
        "doc_id": 4,
        "title": "Western Blot Usage",
        "source": "product_doc",
        "text": "For Western Blot, the recommended dilution is 1:1000 unless otherwise specified."
    }
]

df_kb = pd.DataFrame(kb_data)
df_kb

,doc_id,title,source,text
0,1,Antibody X Storage,product_doc,Antibody X should be stored at -20°C. Avoid re...
1,2,Shipping Policy,company_doc,Most products are shipped on dry ice. Delivery...
2,3,Company Introduction,company_doc,"Promab provides antibodies, proteins, and rela..."
3,4,Western Blot Usage,product_doc,"For Western Blot, the recommended dilution is ..."


In [3]:
class KnowledgeBase:
    def __init__(self, df_kb):
        self.df = df_kb.copy()
        self.vectorizer = TfidfVectorizer(stop_words="english")
        self.doc_matrix = self.vectorizer.fit_transform(self.df["text"].fillna(""))

    def search(self, query, top_k=3):
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.doc_matrix).flatten()

        result_df = self.df.copy()
        result_df["score"] = scores
        result_df = result_df.sort_values("score", ascending=False).head(top_k).reset_index(drop=True)

        return result_df

In [4]:
class SupportAgent:
    def __init__(self, df_kb):
        self.kb = KnowledgeBase(df_kb)

    def classify_question(self, subject, customer_message):
        text = f"{subject} {customer_message}".lower()

        if any(k in text for k in ["storage", "protocol", "dilution", "specification", "technical"]):
            return "technical_question"
        if any(k in text for k in ["shipping", "delivery", "tracking", "dry ice"]):
            return "shipping_question"
        if any(k in text for k in ["invoice", "payment", "billing", "quote", "price"]):
            return "payment_or_quote"
        if any(k in text for k in ["company", "about", "introduction"]):
            return "company_info"

        return "general_question"

    def build_query(self, subject, customer_message, extra_context=""):
        return f"{subject}\n{customer_message}\n{extra_context}".strip()

    def generate_answer_summary(self, retrieved_df):
        if retrieved_df.empty:
            return "I could not find enough information in the knowledge base to answer this confidently."

        top_texts = retrieved_df["text"].head(2).tolist()
        return " ".join(top_texts)

    def generate_reply_draft(self, answer_summary):
        return f"""Dear Customer,

Thank you for your inquiry.

Based on the information we have, {answer_summary}

Please let us know if you have any further questions.

Best regards,
Promab Team
"""

    def compute_confidence(self, retrieved_df):
        if retrieved_df.empty:
            return 0.0
        return round(float(retrieved_df.iloc[0]["score"]), 3)

    def needs_human_review(self, category, confidence):
        if confidence < 0.2:
            return True
        if category in {"payment_or_quote"}:
            return True
        return False

    def run(self, subject, customer_message, extra_context=""):
        category = self.classify_question(subject, customer_message)
        query = self.build_query(subject, customer_message, extra_context)
        retrieved_df = self.kb.search(query, top_k=3)
        answer_summary = self.generate_answer_summary(retrieved_df)
        reply_draft = self.generate_reply_draft(answer_summary)
        confidence = self.compute_confidence(retrieved_df)
        human_review = self.needs_human_review(category, confidence)

        result = {
            "category": category,
            "query": query,
            "retrieved_sources": retrieved_df[["doc_id", "title", "source", "score"]].copy(),
            "answer_summary": answer_summary,
            "reply_draft": reply_draft,
            "confidence": confidence,
            "needs_human_review": human_review
        }

        return result

In [5]:
agent = SupportAgent(df_kb)

In [6]:
subject = "Question about storage condition"
customer_message = "Hi, could you confirm how this antibody should be stored?"
extra_context = ""

result = agent.run(subject, customer_message, extra_context)

In [7]:
print("=== CATEGORY ===")
print(result["category"])

print("\n=== QUERY ===")
print(result["query"])

print("\n=== RETRIEVED SOURCES ===")
display(result["retrieved_sources"])

print("\n=== ANSWER SUMMARY ===")
print(result["answer_summary"])

print("\n=== REPLY DRAFT ===")
print(result["reply_draft"])

print("\n=== CONFIDENCE ===")
print(result["confidence"])

print("\n=== NEEDS HUMAN REVIEW ===")
print(result["needs_human_review"])

=== CATEGORY ===
technical_question

=== QUERY ===
Question about storage condition
Hi, could you confirm how this antibody should be stored?

=== RETRIEVED SOURCES ===


,doc_id,title,source,score
0,1,Antibody X Storage,product_doc,0.5
1,2,Shipping Policy,company_doc,0.0
2,3,Company Introduction,company_doc,0.0



=== ANSWER SUMMARY ===
Antibody X should be stored at -20°C. Avoid repeated freeze-thaw cycles. Most products are shipped on dry ice. Delivery timing depends on destination and carrier.

=== REPLY DRAFT ===
Dear Customer,

Thank you for your inquiry.

Based on the information we have, Antibody X should be stored at -20°C. Avoid repeated freeze-thaw cycles. Most products are shipped on dry ice. Delivery timing depends on destination and carrier.

Please let us know if you have any further questions.

Best regards,
Promab Team


=== CONFIDENCE ===
0.5

=== NEEDS HUMAN REVIEW ===
False
